# Gaussian wave packet: density, real and imaginary parts

*Professor Nano Labs · Quantum foundations*

**Try it live in your browser:** [Gaussian Wave Packet lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/gaussian-wave-packet/)  
**Watch first:** [Free quantum electron wave versus Gaussian packet](https://www.youtube.com/watch?v=7hOyxlRb2LM)  
**Watch later:** [Fourier vs Heisenberg uncertainty principle](https://www.youtube.com/watch?v=jfwoUOOw5ms) · [Schrodinger equation and time evolution of quantum state](https://www.youtube.com/watch?v=gXkbSp2xzlc)

**Run all cells from the top, then press Play below either animation.**

The first animation shows the probability density $|\Psi(x,t)|^2$. The second shows
$\mathrm{Re}\,\Psi(x,t)$ (blue, solid) and $\mathrm{Im}\,\Psi(x,t)$ (orange, dashed)
on the same axes. Both use the same computed complex wavefunction and times; their
playback controls are independent.

The wavefunction is a superposition of free-particle plane waves:

$$\Psi(x,t)=\frac{1}{\sqrt{2\pi}}\int_{-\infty}^{\infty}
\Phi(k)e^{ikx}e^{-i\omega_k t}\,dk.$$

The Gaussian Fourier amplitude, centred at $k_0$, is

$$\Phi(k)=\left(\frac{2}{\pi}\right)^{1/4}\sqrt{\sigma}
\exp[-\sigma^2(k-k_0)^2],\qquad q=k-k_0.$$

The initial state is therefore

$$\Psi(x,0)=(2\pi\sigma^2)^{-1/4}\exp[-x^2/(4\sigma^2)]e^{ik_0x}.$$

Each plane wave evolves with the full free-particle dispersion $\omega_k=\hbar k^2/(2m)$.
A common simplification **linearizes** $\omega_k$ around $k_0$ to explain the group
velocity; that approximation translates the envelope without spreading. It is
deliberately **not** the propagation rule used here.

### Parameters and conventions

$\sigma$ is the initial standard deviation of $|\Psi(x,0)|^2$. If you have seen the
packet written as $\Psi\propto\exp[-x^2/(2\sigma_x^2)]$, then $\sigma_x=\sqrt{2}\,\sigma$;
the default `sigma = 1e-9 / sqrt(2)` corresponds to $\sigma_x = 1$ nm.

The defaults are $m = 0.5\,m_0$ and $k_0 = 0$, a packet at rest that only spreads,
shown in a $[-20, 20]$ nm window from 0 to 50 fs in 50 frames played at 50 ms each.
They match the browser lab's defaults. To send the packet moving to the right, set
`k0 = 3e9`, `sigma = 2e-9`, `m = m0` and widen the window to $\pm 40$ nm.

The quadrature uses a finite $k$ grid, with the cutoff and resolution clearly
marked as numerical implementation choices. The code sums **complex amplitudes**
first and only then forms $|\Psi|^2$. There is no closed-form Gaussian
propagation hidden in the animation.

Only NumPy, Matplotlib, and IPython (normally already available in Jupyter)
are needed. No video encoder or interactive plotting backend is required.


In [ ]:
from __future__ import annotations

import argparse
import re
import webbrowser
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from IPython.display import HTML, display


In [ ]:
# ================================================================
# 1. PARAMETERS
# ================================================================
# All calculations use SI units. Only the plotted axes are converted.
hbar = 1.054571817e-34     # hbar: reduced Planck constant, J s (CODATA 2018)
m0 = 9.1093837015e-31      # Electron rest mass, kg (CODATA 2018)
m = 0.5 * m0               # m: particle mass, kg
k0 = 0.0                   # k_0: center of Phi(k), m^-1. Try 3e9 for a moving packet.

# IMPORTANT WIDTH CONVENTION:
# sigma is the standard deviation of |Psi(x,0)|^2:
#   Psi(x,0) = (2*pi*sigma**2)**(-1/4)
#              * exp[-x**2/(4*sigma**2)] * exp(i*k0*x).
# The carrier exp(i*k0*x) gives the packet an average momentum hbar*k0.
# With k0 = 0 the packet stays centred and only spreads.
# Writing Psi ~ exp[-x**2/(2*sigma_x**2)] instead gives sigma_x = sqrt(2)*sigma,
# so this default corresponds to sigma_x = 1 nm.
sigma = 1.0e-9 / np.sqrt(2.0)   # sigma = 0.70710678 nm, NOT 1 nm

# Position window, time span and frame count (the browser lab uses the same defaults).
# The initial packet is centered at x = 0.
xmin, xmax = -20.0e-9, 20.0e-9  # m; try -50e-9, 50e-9 for a wider view
Nx = 1000
x = np.linspace(xmin, xmax, Nx)

tmin, tmax = 0.0, 50.0e-15      # s: 0 to 50 fs
Nt = 50
times = np.linspace(tmin, tmax, Nt)
interval_ms = 50                # Playback speed, not physical time step

# Numerical integration settings (implementation choices, not physical inputs).
Nk = 2049                      # Number of sampled plane-wave components
q_cutoff = 8.0 / sigma          # At endpoints, Phi/Phi_peak = exp(-64)

if not (m > 0 and sigma > 0 and xmin < xmax and tmin <= tmax):
    raise ValueError("Require m > 0, sigma > 0, xmin < xmax, tmin <= tmax.")
if Nx < 2 or Nk < 3 or Nt < 2 or interval_ms <= 0:
    raise ValueError("Require Nx >= 2, Nk >= 3, Nt >= 2, interval_ms > 0.")


In [ ]:
# ================================================================
# 2. Phi(k), E_k, omega_k
# ================================================================
def Phi(k: np.ndarray) -> np.ndarray:
    """Gaussian Fourier amplitude centered at k0.

    For k0 = 0 it is (2/pi)**(1/4) * sqrt(sigma) * exp(-sigma**2*k**2).
    Replacing k with q = k-k0 centers that spectrum at k0.
    This corresponds to multiplying the initial spatial Gaussian by
    exp(i*k0*x); it does not change its probability density.
    """
    q = np.asarray(k, dtype=float) - k0   # q = k - k_0
    return (2.0 / np.pi)**0.25 * np.sqrt(sigma) * np.exp(-sigma**2 * q**2)


q = np.linspace(-q_cutoff, q_cutoff, Nk)
k = k0 + q
# dk is the integration step Delta k; it is NOT a wavevector uncertainty.
dk = k[1] - k[0]
Phi_k = Phi(k)
E_k = hbar**2 * k**2 / (2.0 * m)     # E_k = hbar^2 k^2 / (2m)
omega_k = E_k / hbar                # omega_k = hbar k^2 / (2m)

# Trapezoidal quadrature: endpoints have half weight.
weights = np.ones(Nk)
weights[0] = weights[-1] = 0.5

# IMPORTANT: We use the FULL quadratic dispersion above.
# The common linearized form uses omega_k ~ omega_k0 + omega'_k0*q.
# That FIRST-ORDER approximation describes translation without spreading.
# It is NOT used here. The full plane-wave integral is used instead.


# ================================================================
# 3. Psi(x,t) — DIRECT PLANE-WAVE SUM
# ================================================================
def Psi(x: np.ndarray, t: float) -> np.ndarray:
    """Compute the complex wavefunction on a 1D position grid, at time t.

    The plane-wave superposition:
        Psi(x,t) = 1/sqrt(2*pi) * integral dk
                   Phi(k) * exp(i*k*x) * exp(-i*omega_k*t).

    Here the integral is approximated with a finite, sufficiently wide
    k grid and trapezoidal quadrature. This is NOT a fitted Gaussian,
    an intensity-only propagation, or a closed-form envelope formula.
    The complex phase of EVERY plane wave is retained before summing.
    """
    x = np.asarray(x, dtype=float)
    if x.ndim != 1 or not np.all(np.isfinite(x)):
        raise ValueError("x must be a finite, one-dimensional position array.")
    if np.ndim(t) != 0 or not np.isfinite(t):
        raise ValueError("t must be a finite scalar time in seconds.")

    # Rows are positions x_i; columns are wavevectors k_j.
    exp_ikx = np.exp(1j * x[:, None] * k[None, :])
    exp_minus_i_omega_kt = np.exp(-1j * omega_k * t)

    # Each column is one evolving plane wave Psi_k(x,t)
    # = exp(i*k*x) * exp(-i*omega_k*t), a free-particle plane wave.
    Psi_k_xt = exp_ikx * exp_minus_i_omega_kt[None, :]

    # >>> THIS IS THE SUMMATION OF PLANE WAVES. <<<
    # Psi(x_i,t) ~= dk/sqrt(2*pi) * SUM_j
    #               weights[j] * Phi(k_j) * Psi_{k_j}(x_i,t).
    # axis=1 sums over k_j, leaving one complex value per position x_i.
    Psi_xt = dk / np.sqrt(2.0 * np.pi) * np.sum(
        weights[None, :] * Phi_k[None, :] * Psi_k_xt,
        axis=1,
    )
    return Psi_xt


def compute_evolution() -> np.ndarray:
    """Evaluate the plane-wave sum at each time; return complex Psi_xt[time, position]."""
    # Each time is computed from the original Phi(k), not advanced by
    # a finite-difference time step. Changing Nt affects animation sampling.
    return np.stack([Psi(x, float(t)) for t in times])


def diagnostics(Psi_xt: np.ndarray) -> dict[str, float]:
    """Report spectral norm and the probability inside the displayed window."""
    norm_k = float(dk * np.sum(weights * np.abs(Phi_k)**2))
    x_weights = np.ones(x.size)
    x_weights[0] = x_weights[-1] = 0.5
    probability_in_window = (x[1] - x[0]) * np.sum(
        x_weights[None, :] * np.abs(Psi_xt)**2, axis=1
    )
    report = {
        "spectral_norm": norm_k,
        "initial_window_probability": float(probability_in_window[0]),
        "final_window_probability": float(probability_in_window[-1]),
    }
    print(f"Computed {Nt} frames using {Nk} plane waves per position.")
    print(f"sigma = {sigma * 1e9:.8f} nm; k0 = {k0 * 1e-9:.3f} nm^-1")
    print(f"k-space normalization: integral |Phi(k)|^2 dk = {norm_k:.12f}")
    if np.min(probability_in_window) < 0.99:
        print(
            f"At t = {times[-1] * 1e15:g} fs, the plotted window contains "
            f"{100 * probability_in_window[-1]:.2f}% of the probability. "
            "Spreading tails extend outside the axes; the wavefunction is "
            "NOT renormalized within this window. Widen xmin/xmax to see them."
        )
    return report


## Where the plane waves are summed

Inside `Psi(x, t)`, the explicitly marked `np.sum(..., axis=1)` evaluates

$$\Psi(x_i,t)\approx\frac{\Delta k}{\sqrt{2\pi}}
\sum_j w_j\Phi(k_j)e^{ik_jx_i}e^{-i\omega_{k_j}t},$$

where $w_j=1$ except for the two endpoint weights $w_j=1/2$.
These weights implement the trapezoidal rule. `dk` is an integration step,
not a wavevector uncertainty.

The next cell computes every frame. It does **not** renormalize each frame
inside the plotted position window. At late times some probability lies
outside the original axes; that is clipping, not probability loss.


In [ ]:
Psi_xt = compute_evolution()
report = diagnostics(Psi_xt)


In [ ]:
# ================================================================
# 4. PROBABILITY-DENSITY ANIMATION, WITH PLAYBACK CONTROLS
# ================================================================
def make_animation(Psi_xt: np.ndarray) -> tuple[plt.Figure, FuncAnimation]:
    """Animate |Psi(x,t)|^2, where Psi was computed by the plane-wave sum."""
    if Psi_xt.shape != (times.size, x.size):
        raise ValueError("Psi_xt must have shape (number of times, number of x points).")
    if not np.all(np.isfinite(Psi_xt)):
        raise ValueError("The computed wavefunction contains nonfinite values.")

    # Take the absolute square ONLY AFTER summing the complex amplitudes.
    # Summing individual plane-wave intensities would lose interference.
    rho_nm = np.abs(Psi_xt)**2 * 1e-9  # m^-1 -> nm^-1
    x_nm = x * 1e9                    # m -> nm

    fig, ax = plt.subplots(figsize=(10, 6), dpi=100)
    line, = ax.plot(x_nm, rho_nm[0], linewidth=2)
    ax.set_xlim(xmin * 1e9, xmax * 1e9)
    ax.set_ylim(0, 1.1 * rho_nm.max())
    ax.set_xlabel("Position x (nm)", fontsize=13)
    ax.set_ylabel(r"Probability density $|\Psi(x,t)|^2$ (nm$^{-1}$)", fontsize=13)
    ax.set_title("Time Evolution of a Gaussian Wave Packet", fontsize=15)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=12)
    time_text = ax.text(
        0.97, 0.93, "", transform=ax.transAxes, ha="right", fontsize=13
    )
    fig.tight_layout()

    def animate(i: int):
        line.set_ydata(rho_nm[i])
        time_text.set_text(f"t = {times[i] * 1e15:.1f} fs")
        return line, time_text

    def init():
        return animate(0)

    ani = FuncAnimation(
        fig, animate, init_func=init, frames=times.size,
        interval=interval_ms, blit=True, repeat=True,
    )
    return fig, ani


# ================================================================
# 6. HTML PLAYBACK HELPERS — shared by both animations
# ================================================================
def playback_html(ani: FuncAnimation) -> str:
    """Create embedded HTML with play, pause, reverse, slider, and speed controls.

    Plain-text buttons replace font icons so controls work offline too.
    No ffmpeg, video player package, or Jupyter widget backend is needed.
    """
    with plt.rc_context({"animation.embed_limit": 100.0}):
        fragment = ani.to_jshtml(
            fps=1000.0 / interval_ms, embed_frames=True, default_mode="loop"
        )

    # Remove the external icon-font dependency; preserve Matplotlib's JS.
    fragment = re.sub(
        r'<link\b[^>]*font-awesome[^>]*>', "", fragment, flags=re.IGNORECASE
    )
    buttons = {
        "fa fa-minus": "&#8722;",
        "fa fa-fast-backward": "First",
        "fa fa-step-backward": "Prev",
        "fa fa-play fa-flip-horizontal": "Reverse",
        "fa fa-pause": "Pause",
        "fa fa-play": "Play",
        "fa fa-step-forward": "Next",
        "fa fa-fast-forward": "Last",
        "fa fa-plus": "+",
    }
    for icon_class, label in buttons.items():
        fragment = fragment.replace(f'<i class="{icon_class}"></i>', label)

    # Keep the same controls, with room for readable text button labels.
    fragment += """
<style>
.animation {max-width: 100%;}
.animation > img {max-width: 100%; height: auto;}
.anim-buttons button {width: auto; min-width: 42px; padding: 4px 8px;
                      font: inherit; cursor: pointer;}
</style>
"""
    return fragment


def save_html(
    fragment: str,
    output: Path,
    report: dict[str, float],
    real_imag_fragment: str = "",
) -> Path:
    """Write offline HTML with density and, optionally, Re/Im animations."""
    output = Path(output).expanduser().resolve()
    output.parent.mkdir(parents=True, exist_ok=True)
    tail_note = ""
    if report["final_window_probability"] < 0.99:
        tail_note = (
            f"<p><strong>Position window:</strong> The axes span "
            f"{xmin * 1e9:g} to {xmax * 1e9:g} nm. At "
            f"{times[-1] * 1e15:g} fs they contain "
            f"{100 * report['final_window_probability']:.2f}% of the probability; "
            "the remaining spreading tails lie outside the view. "
            "The curve is not renormalized to this window.</p>"
        )
    sections = (
        '<section id="probability-density">\n'
        '<h2>1. Probability density |Ψ(x,t)|²</h2>\n'
        + fragment + "\n</section>\n"
    )
    if real_imag_fragment:
        sections += (
            '<section id="real-imaginary-parts">\n'
            '<h2>2. Real and imaginary parts of Ψ(x,t)</h2>\n'
            '<p><strong>Blue, solid:</strong> Re Ψ(x,t). '
            '<strong>Orange, dashed:</strong> Im Ψ(x,t). '
            'Both come from the same complex plane-wave sum as the density above. '
            'Amplitudes are displayed in nm<sup>−1/2</sup>, with a fixed '
            'symmetric vertical scale. Each animation has its own playback controls.</p>\n'
            + real_imag_fragment + "\n</section>\n"
        )
    document = """<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Gaussian wave packet — density, real and imaginary parts</title>
<style>body {font-family: system-ui, sans-serif; max-width: 1040px;
margin: 2rem auto; padding: 0 1rem;} p {line-height: 1.5;}</style>
</head><body>
<h1>Gaussian wave packet</h1>
<p>Direct plane-wave superposition, with the full free-particle dispersion
ω<sub>k</sub> = ℏk<sup>2</sup>/(2m). Press <strong>Play</strong>, or use the
slider to inspect a particular time. The default playback mode is loop.</p>
""" + sections + tail_note + "\n</body></html>\n"
    output.write_text(document, encoding="utf-8")
    return output




In [ ]:
fig, ani = make_animation(Psi_xt)
try:
    animation_html = playback_html(ani)
finally:
    plt.close(fig)  # Avoid a duplicate static plot in the notebook.
display(HTML(animation_html))


## 5. Real and imaginary parts of the wavefunction

The next animation plots **$\mathrm{Re}\,\Psi(x,t)$ in blue (solid)** and
**$\mathrm{Im}\,\Psi(x,t)$ in orange (dashed)** on the same axes. It retains
both positive and negative values, with a fixed symmetric vertical scale.

It uses the **same complex array `Psi_xt`** computed above by summing the plane
waves. There is no second evolution model, phase adjustment, or
renormalization. The position window, 0–50 fs time range, frame count, and
playback controls match the probability-density animation.

Only display units change: `Psi_nm = Psi_xt * np.sqrt(1e-9)` converts the
wavefunction from $\mathrm{m}^{-1/2}$ to $\mathrm{nm}^{-1/2}$. Consequently,

$$|\Psi_{\mathrm{nm}}|^2 =
(\mathrm{Re}\,\Psi_{\mathrm{nm}})^2 +
(\mathrm{Im}\,\Psi_{\mathrm{nm}})^2$$

reproduces the density in $\mathrm{nm}^{-1}$ shown above. Unlike the density,
the two curves display the real and imaginary **amplitudes**, not their squares.

Press **Play** below. The two animations have independent controls; use the
same slider position to compare a particular time.

In [ ]:
# ================================================================
# 5. REAL AND IMAGINARY PARTS — same complex plane-wave superposition
# ================================================================
def make_real_imag_animation(
    Psi_xt: np.ndarray,
) -> tuple[plt.Figure, FuncAnimation]:
    """Animate Re[Psi(x,t)] and Im[Psi(x,t)] on the same axes.

    Reuse the COMPLEX Psi_xt computed by the explicit plane-wave sum.
    No extra propagation, absolute value, phase rotation, or
    frame-by-frame renormalization is applied to the wavefunction.
    """
    if Psi_xt.shape != (times.size, x.size):
        raise ValueError("Psi_xt must have shape (number of times, number of x points).")
    if not np.all(np.isfinite(Psi_xt)):
        raise ValueError("The computed wavefunction contains nonfinite values.")

    x_nm = x * 1e9

    # Psi has units m^(-1/2), unlike |Psi|^2, which has units m^(-1).
    # Convert amplitudes to nm^(-1/2), so their squared magnitudes
    # reproduce the density in nm^(-1) shown in the first animation:
    #   |Psi_nm|^2 = (Re Psi_nm)^2 + (Im Psi_nm)^2
    #              = |Psi_xt|^2 * 1e-9.
    Psi_nm = Psi_xt * np.sqrt(1e-9)

    # Extract the real and imaginary parts AFTER the plane-wave sum.
    # Do NOT take np.abs() here: both curves retain their signs.
    Re_Psi = np.real(Psi_nm)
    Im_Psi = np.imag(Psi_nm)

    fig, ax = plt.subplots(figsize=(10, 6), dpi=100)
    line_re, = ax.plot(
        x_nm, Re_Psi[0], color="tab:blue", linestyle="-", linewidth=2,
        label=r"$\mathrm{Re}\,\Psi(x,t)$",
    )
    line_im, = ax.plot(
        x_nm, Im_Psi[0], color="tab:orange", linestyle="--", linewidth=2,
        label=r"$\mathrm{Im}\,\Psi(x,t)$",
    )
    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.5, zorder=0)
    ax.set_xlim(xmin * 1e9, xmax * 1e9)

    # Fix one symmetric vertical scale for the entire animation.
    # This shows amplitude changes without hiding them by rescaling.
    amplitude_limit = 1.1 * float(np.max(np.abs(Psi_nm)))
    if amplitude_limit == 0.0:
        amplitude_limit = 1.0
    ax.set_ylim(-amplitude_limit, amplitude_limit)
    ax.set_xlabel("Position x (nm)", fontsize=13)
    ax.set_ylabel(r"Wavefunction amplitude (nm$^{-1/2}$)", fontsize=13)
    ax.set_title("Gaussian Wave Packet: Real and Imaginary Parts", fontsize=15)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=12)
    ax.legend(loc="upper left", fontsize=13)
    time_text = ax.text(
        0.97, 0.93, "", transform=ax.transAxes, ha="right", fontsize=13
    )
    fig.tight_layout()

    def animate(i: int):
        # Same physical time and same Psi_xt[i] as the density animation.
        line_re.set_ydata(Re_Psi[i])
        line_im.set_ydata(Im_Psi[i])
        time_text.set_text(f"t = {times[i] * 1e15:.1f} fs")
        return line_re, line_im, time_text

    def init():
        return animate(0)

    ani = FuncAnimation(
        fig, animate, init_func=init, frames=times.size,
        interval=interval_ms, blit=True, repeat=True,
    )
    return fig, ani

In [ ]:
fig_ri, ani_ri = make_real_imag_animation(Psi_xt)
try:
    real_imag_html = playback_html(ani_ri)
finally:
    plt.close(fig_ri)  # Avoid a duplicate static plot in the notebook.
display(HTML(real_imag_html))


### Optional: save both animations for browser playback

Uncomment and run the next cell to write a standalone HTML page containing
both animations and their playback controls. It works offline in a browser.

To see more of the spreading tails, widen `xmin` and `xmax` in the parameter
cell and run all cells again. Increase `Nk` when extending the time range
or position window substantially, and check convergence.


In [ ]:
# output = save_html(
#     animation_html,
#     Path("gaussian_packet_evolution_fourier_real_imag.html"),
#     report,
#     real_imag_fragment=real_imag_html,
# )
# print(output)


## Things to try

Each idea matches a "Predict first" question in the [browser lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/gaussian-wave-packet/). Edit the
parameter cell, then **Run All**.

1. **Narrower start.** Halve `sigma`. After 50 fs, is the packet narrower or wider than before?
2. **Heavier particle.** Set `m = 2 * m0`. How much longer does the spreading take?
3. **Moving packet.** Set `k0 = 3e9`, `sigma = 2e-9`, `m = m0` and widen the window to ±40 nm.
   Do the crests of $\mathrm{Re}\,\Psi$ travel with the packet?
4. **Check the spreading law.** Measure the width of $|\Psi|^2$ at a few times and compare it with
   $\sigma(t)=\sigma\sqrt{1+(t/\tau)^2}$, where $\tau=2m\sigma^2/\hbar$.
